<a href="https://www.kaggle.com/code/robiulhasanjisan/stacking-ensemble?scriptVersionId=307758237" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.ensemble import StackingRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error

import joblib
import optuna
from scipy import stats


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12



In [ ]:

# Load data
DATA_PATH = "/kaggle/input/university-student-performance-and-habits-dataset/Student_data.csv"
df = pd.read_csv(DATA_PATH)
TARGET = "Final_CGPA"


In [ ]:


print(f"\nDataset Shape: {df.shape}")
print(f"Memory Usage: {df.memory_usage().sum() / 1024**2:.2f} MB")
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"\nData Types:\n{df.dtypes}")

#  TARGET VARIABLE DISTRIBUTION

In [ ]:




fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram with KDE
axes[0, 0].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(df[TARGET].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[TARGET].mean():.2f}')
axes[0, 0].axvline(df[TARGET].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df[TARGET].median():.2f}')
axes[0, 0].set_title(f'{TARGET} Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel(TARGET)
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Box plot
sns.boxplot(y=df[TARGET], ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title(f'{TARGET} Box Plot', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel(TARGET)

# Q-Q plot for normality check
from scipy import stats
stats.probplot(df[TARGET], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot for Normality Check', fontsize=14, fontweight='bold')

# Summary statistics table
stats_text = f"""Summary Statistics:
Mean: {df[TARGET].mean():.4f}
Median: {df[TARGET].median():.4f}
Std Dev: {df[TARGET].std():.4f}
Skewness: {df[TARGET].skew():.4f}
Kurtosis: {df[TARGET].kurtosis():.4f}
Min: {df[TARGET].min():.4f}
Max: {df[TARGET].max():.4f}"""
axes[1, 1].text(0.1, 0.5, stats_text, transform=axes[1, 1].transAxes, 
                fontsize=12, verticalalignment='center', 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


#  CORRELATION ANALYSIS

In [ ]:



# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Top correlations with target
corr_with_target = corr_matrix[TARGET].sort_values(ascending=False)
print(f"\nTop Correlations with {TARGET}:")
print(corr_with_target)


# FEATURE ANALYSIS VISUALIZATIONS

In [ ]:



# Create subplots for key features vs target
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
features_to_plot = ['Previous_CGPA', 'Study_Hours_Per_Day', 'Attendance_Pct', 
                    'Sleep_Hours', 'Social_Hours_Week', 'Age']

for idx, feature in enumerate(features_to_plot):
    row, col = idx // 3, idx % 3
    axes[row, col].scatter(df[feature], df[TARGET], alpha=0.5, s=10)
    
    # Add trend line
    z = np.polyfit(df[feature], df[TARGET], 1)
    p = np.poly1d(z)
    axes[row, col].plot(df[feature].sort_values(), p(df[feature].sort_values()), 
                        "r-", linewidth=2, label=f'Trend: r={corr_matrix.loc[feature, TARGET]:.3f}')
    
    axes[row, col].set_xlabel(feature, fontsize=12)
    axes[row, col].set_ylabel(TARGET, fontsize=12)
    axes[row, col].set_title(f'{feature} vs {TARGET}', fontsize=13, fontweight='bold')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# CATEGORICAL VARIABLES ANALYSIS

In [ ]:

# Gender analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Box plot by gender
sns.boxplot(data=df, x='Gender', y=TARGET, ax=axes[0], palette='Set2')
axes[0].set_title(f'{TARGET} by Gender', fontsize=14, fontweight='bold')
axes[0].set_ylabel(TARGET)

# Count plot
sns.countplot(data=df, x='Gender', ax=axes[1], palette='Set2')
axes[1].set_title('Gender Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(df['Gender'].value_counts()):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Violin plot
sns.violinplot(data=df, x='Gender', y=TARGET, ax=axes[2], palette='Set2')
axes[2].set_title(f'{TARGET} Distribution by Gender', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Major analysis
top_majors = df['Major'].value_counts().head(10)
plt.figure(figsize=(14, 8))

# Box plot for top majors
sns.boxplot(data=df[df['Major'].isin(top_majors.index)], 
            x='Major', y=TARGET, order=top_majors.index)
plt.title(f'{TARGET} by Major (Top 10)', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.ylabel(TARGET)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#  STUDENT SEGMENTATION ANALYSIS

In [ ]:





# Create performance segments
df['Performance_Segment'] = pd.cut(df[TARGET], 
                                   bins=[0, 2.0, 2.5, 3.0, 3.5, 4.0],
                                   labels=['Very Low (<2.0)', 'Low (2.0-2.5)', 
                                           'Medium (2.5-3.0)', 'High (3.0-3.5)', 
                                           'Very High (3.5-4.0)'])

# Radar chart for feature patterns across segments
fig = plt.figure(figsize=(12, 10))
features_for_radar = ['Study_Hours_Per_Day', 'Attendance_Pct', 'Sleep_Hours', 'Previous_CGPA']
radar_data = df.groupby('Performance_Segment')[features_for_radar].mean()

# Normalize data for radar chart
radar_data_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

angles = np.linspace(0, 2*np.pi, len(features_for_radar), endpoint=False).tolist()
radar_data_norm = radar_data_norm.reset_index()
angles += angles[:1]

ax = plt.subplot(111, projection='polar')
for i, segment in enumerate(radar_data_norm['Performance_Segment']):
    values = radar_data_norm.iloc[i, 1:].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=segment)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features_for_radar)
ax.set_title('Feature Patterns by Performance Segment', size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.show()



#  INTERACTIVE VISUALIZATIONS (Plotly)

In [ ]:


# 3D Scatter plot
fig = px.scatter_3d(df, x='Study_Hours_Per_Day', y='Attendance_Pct', z=TARGET,
                    color='Gender', size='Previous_CGPA', hover_name='Major',
                    title='3D Visualization: Study Hours vs Attendance vs CGPA',
                    labels={'Study_Hours_Per_Day': 'Study Hours/Day', 
                            'Attendance_Pct': 'Attendance %',
                            TARGET: 'Final CGPA'})
fig.update_layout(title_font_size=16, title_font_weight='bold')
fig.show()

# Parallel coordinates plot
fig = px.parallel_coordinates(df, color=TARGET,
                             dimensions=['Previous_CGPA', 'Study_Hours_Per_Day', 
                                       'Attendance_Pct', 'Sleep_Hours', 'Social_Hours_Week'],
                             color_continuous_scale=px.colors.diverging.RdYlGn,
                             title='Parallel Coordinates: Feature Relationships with Final CGPA')
fig.update_layout(title_font_size=16, title_font_weight='bold')
fig.show()


# DATA READY FOR ML 

In [ ]:
DATA_PATH = "/kaggle/input/university-student-performance-and-habits-dataset/Student_data.csv"
df = pd.read_csv(DATA_PATH)
# Separate features and target
X = df.drop(columns=["Student_ID", TARGET])
y = df[TARGET]

# Identify column types
categorical_cols = ["Gender", "Major"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

print(f"\nNumerical Features: {numerical_cols}")
print(f"Categorical Features: {categorical_cols}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pd.cut(y, bins=5)
)

print(f"\nTraining Set: {X_train.shape[0]} samples")
print(f"Test Set: {X_test.shape[0]} samples")
print(f"Target Mean (Train): {y_train.mean():.4f} ± {y_train.std():.4f}")
print(f"Target Mean (Test): {y_test.mean():.4f} ± {y_test.std():.4f}")

#  PREPROCESSING PIPELINE

In [ ]:

# Advanced preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]
)

# Create polynomial features option
poly_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]
)


#  BASELINE MODELS

In [ ]:


def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, 
                                scoring='r2', n_jobs=-1)
    
    # Train on full training data
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    evs = explained_variance_score(y_test, y_pred)
    
    print(f"\n{model_name}:")
    print(f"  CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test R²: {r2:.4f}")
    print(f"  Test MAE: {mae:.4f}")
    print(f"  Test RMSE: {rmse:.4f}")
    print(f"  Test MAPE: {mape:.2f}%")
    print(f"  Explained Variance: {evs:.4f}")
    
    return {'model': model, 'name': model_name, 'r2': r2, 'mae': mae, 
            'rmse': rmse, 'mape': mape, 'cv_mean': cv_scores.mean()}

# List of baseline models
baseline_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Bayesian Ridge': BayesianRidge(),
    'KNN': KNeighborsRegressor(n_neighbors=10),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_state=42, verbose=0, allow_writing_files=False),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42)
}

# Evaluate baseline models
baseline_results = []
for name, model in baseline_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    result = evaluate_model(pipeline, X_train, y_train, X_test, y_test, name)
    baseline_results.append(result)

# Create results dataframe
results_df = pd.DataFrame(baseline_results)
results_df = results_df.sort_values('r2', ascending=False)

print("BASELINE MODELS RANKING")

print(results_df[['name', 'r2', 'mae', 'rmse', 'mape', 'cv_mean']].to_string(index=False))


#  ENSEMBLE MODELS

In [ ]:
best_xgb = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
best_lgbm = LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
best_cat = CatBoostRegressor(iterations=100, random_state=42, verbose=0, allow_writing_files=False)

In [ ]:
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

# ============================
# 1. Define Ensembles
# ============================

# Voting Regressor
voting_regressor = VotingRegressor(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('cat', best_cat)
    ],
    weights=[1, 1, 1]
)

# Stacking Regressor with meta-learner
stacking_regressor = StackingRegressor(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('cat', best_cat),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ],
    final_estimator=Ridge(alpha=0.5),
    cv=5,
    n_jobs=-1,
    passthrough=True
)

# Weighted Voting Regressor
weighted_ensemble = VotingRegressor(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('cat', best_cat)
    ],
    weights=[2, 2, 1]
)

# ============================
# 2. Evaluate Ensembles
# ============================

ensembles = {
    'Voting Regressor': voting_regressor,
    'Weighted Ensemble': weighted_ensemble,
    'Stacking Regressor': stacking_regressor
}

ensemble_results = []

for name, model in ensembles.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    result = evaluate_model(pipeline, X_train, y_train, X_test, y_test, name)
    ensemble_results.append(result)

# ============================
# 3. Combine Baseline + Ensemble Results
# ============================

all_results = baseline_results + ensemble_results
all_results_df = pd.DataFrame(all_results)
all_results_df = all_results_df.sort_values('r2', ascending=False).reset_index(drop=True)

# ============================
# 4. Select Best Model
# ============================

best_row = all_results_df.iloc[0].copy()  # safe copy

best_model = best_row['model']            # pipeline with preprocessor + model
best_model_name = best_row['name']
best_model_info = best_row                 # metrics info

print(f"\n Best Model Selected: {best_model_name}")

# ============================
# 5. Display Top 10 Models
# ============================

print("\nFINAL MODEL RANKING (Top 10):")
print(all_results_df[['name', 'r2', 'mae', 'rmse', 'mape', 'cv_mean']].head(10).to_string(index=False))

#  BEST MODEL ANALYSIS

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Style setup
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (18, 12)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Use best model directly (IMPORTANT FIX)
final_pipeline = best_model
final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_pred = np.clip(y_pred, 0.0, 4.0)

residuals = y_test - y_pred

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

#  Color palette
main_color = "#4C72B0"
accent_color = "#DD8452"
green_color = "#55A868"


# 1. Actual vs Predicted

axes[0, 0].scatter(
    y_test, y_pred, 
    alpha=0.6, color=main_color, edgecolor='black', linewidth=0.5
)

axes[0, 0].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle='--', color=accent_color, linewidth=2
)

axes[0, 0].set_title(f'Actual vs Predicted\nR² = {best_model_info["r2"]:.4f}', fontweight='bold')
axes[0, 0].set_xlabel('Actual CGPA')
axes[0, 0].set_ylabel('Predicted CGPA')


# 2. Residual Plot

sns.scatterplot(
    x=y_pred, y=residuals,
    ax=axes[0, 1],
    color=main_color,
    edgecolor='black'
)

axes[0, 1].axhline(0, linestyle='--', color=accent_color, linewidth=2)
axes[0, 1].set_title('Residual Plot', fontweight='bold')
axes[0, 1].set_xlabel('Predicted CGPA')
axes[0, 1].set_ylabel('Residuals')


# 3. Residual Distribution

sns.histplot(
    residuals,
    bins=30,
    kde=True,
    ax=axes[0, 2],
    color=green_color
)

axes[0, 2].axvline(0, linestyle='--', color=accent_color, linewidth=2)
axes[0, 2].set_title('Residual Distribution', fontweight='bold')


# 4. Q-Q Plot

stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].get_lines()[0].set_markerfacecolor(main_color)
axes[1, 0].get_lines()[1].set_color(accent_color)
axes[1, 0].set_title('Q-Q Plot of Residuals', fontweight='bold')


# 5. Error Boxplot

sns.boxplot(
    y=np.abs(residuals),
    ax=axes[1, 1],
    color=main_color
)

axes[1, 1].set_title('Absolute Error Distribution', fontweight='bold')
axes[1, 1].set_ylabel('Absolute Error')


# 6. Feature Importance

if hasattr(best_model.named_steps['model'], 'feature_importances_') or \
   hasattr(best_model.named_steps['model'], 'coef_'):

    model_step = best_model.named_steps['model']

    # Get feature names
    feature_names = []
    for name, trans, cols in preprocessor.transformers_:
        if name == 'num':
            feature_names.extend(cols)
        elif name == 'cat':
            cat_encoder = trans.named_steps['onehot']
            cat_features = cat_encoder.get_feature_names_out(cols)
            feature_names.extend(cat_features)

    if hasattr(model_step, 'feature_importances_'):
        importances = model_step.feature_importances_
    else:
        importances = np.abs(model_step.coef_)

    indices = np.argsort(importances)[-10:]

    sns.barplot(
        x=importances[indices],
        y=[feature_names[i] for i in indices],
        ax=axes[1, 2],
        palette="Blues_d"
    )

    axes[1, 2].set_title('Top 10 Feature Importances', fontweight='bold')
    axes[1, 2].set_xlabel('Importance')

else:
    axes[1, 2].text(0.5, 0.5, 'Feature Importance Not Available',
                    ha='center', va='center', fontsize=12)
    axes[1, 2].set_title('Feature Importance')


# Final Layout

plt.suptitle(f'Model Evaluation Dashboard: {best_model_name}', fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

#  CROSS-VALIDATION ON BEST MODEL

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
# Ensure DataFrame
if isinstance(X_train, np.ndarray):
    X_train = pd.DataFrame(X_train, columns=original_feature_names)

if isinstance(y_train, np.ndarray):
    y_train = pd.Series(y_train)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_r2 = []
cv_mae = []
cv_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    #  Use best_model directly (already pipeline)
    fold_model = best_model
    
    fold_model.fit(X_fold_train, y_fold_train)
    y_pred = fold_model.predict(X_fold_val)
    
    cv_r2.append(r2_score(y_fold_val, y_pred))
    cv_mae.append(mean_absolute_error(y_fold_val, y_pred))
    cv_rmse.append(np.sqrt(mean_squared_error(y_fold_val, y_pred)))
    
    print(f"Fold {fold:2d}: R² = {cv_r2[-1]:.4f}, MAE = {cv_mae[-1]:.4f}, RMSE = {cv_rmse[-1]:.4f}")

print("\n 10-Fold CV Summary:")
print(f"R²   : {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"MAE  : {np.mean(cv_mae):.4f} ± {np.std(cv_mae):.4f}")
print(f"RMSE : {np.mean(cv_rmse):.4f} ± {np.std(cv_rmse):.4f}")

#  SAVE BEST MODEL

In [ ]:


# Save the final pipeline
model_filename = f"best_student_gpa_model_{best_model_name.replace(' ', '_')}.pkl"
joblib.dump(final_pipeline, model_filename)
print(f"Model saved as: {model_filename}")

# Save preprocessing and metadata
metadata = {
    'model_name': best_model_name,
    'features': X.columns.tolist(),
    'categorical_cols': categorical_cols,
    'numerical_cols': numerical_cols,
    'target': TARGET,
    'train_shape': X_train.shape,
    'test_shape': X_test.shape,
    'performance': {
        'r2': best_model_info['r2'],
        'mae': best_model_info['mae'],
        'rmse': best_model_info['rmse'],
        'mape': best_model_info['mape'],
        'cv_mean': best_model_info['cv_mean']
    },
    'cv_results': {
        'r2_mean': np.mean(cv_r2),
        'r2_std': np.std(cv_r2),
        'mae_mean': np.mean(cv_mae),
        'mae_std': np.std(cv_mae),
        'rmse_mean': np.mean(cv_rmse),
        'rmse_std': np.std(cv_rmse)
    }
}

joblib.dump(metadata, f"model_metadata_{best_model_name.replace(' ', '_')}.pkl")
print(" saved successfully!")

#  PREDICTION FUNCTION

In [ ]:


def predict_student_gpa(model, student_data):
    """
    Predict student's final CGPA
    
    Parameters:
    -----------
    model: trained pipeline
    student_data: dict or DataFrame with student features
    
    Returns:
    --------
    prediction: float (0-4 scale)
    """
    if isinstance(student_data, dict):
        student_df = pd.DataFrame([student_data])
    else:
        student_df = student_data
    
    prediction = model.predict(student_df)
    prediction = np.clip(prediction, 0.0, 4.0)
    
    return prediction[0]

# Example prediction
sample_student = {
    'Gender': 'Male',
    'Age': 21,
    'Major': 'Computer Science',
    'Attendance_Pct': 85.0,
    'Study_Hours_Per_Day': 6.0,
    'Previous_CGPA': 3.2,
    'Sleep_Hours': 7.0,
    'Social_Hours_Week': 5
}

predicted_gpa = predict_student_gpa(final_pipeline, sample_student)

print(f"  Features: {sample_student}")
print(f"  Predicted Final CGPA: {predicted_gpa:.3f}")
